# SARS plotting workflow

This notebook loads CSV outputs from `run_for_sars.py` and generates reorganized summary figures.

## Imports and repo discovery

In [1]:
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start):
    for path in [start, *start.parents]:
        if (path / "data_sars").exists() and (path / "scripts" / "sars_run").exists():
            return path
    raise RuntimeError("Could not locate Net_Dyns_Project_EpiRank repo root")


repo_root = find_repo_root(Path.cwd().resolve())
scripts_dir = repo_root / "scripts"
sars_run_dir = scripts_dir / "sars_run"
for path in [scripts_dir, sars_run_dir]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

import plot_sars_results as plot

## Load result tables

In [2]:
results_dir = repo_root / "results" / "sars"
movement_df, baseline_df, sensitivity_df, top_scores_df = plot.load_results(results_dir)

{
    "movement_rows": len(movement_df),
    "baseline_rows": len(baseline_df),
    "sensitivity_rows": len(sensitivity_df),
    "top_score_rows": len(top_scores_df),
}

{'movement_rows': 3,
 'baseline_rows': 3,
 'sensitivity_rows': 420,
 'top_score_rows': 30}

## Preview the result tables

In [3]:
movement_df

,metric,pearson,pearson_p,spearman,spearman_p
0,backward_only,0.432563,2.136811e-03,0.451060,1.292828e-03
1,bidirectional,0.689039,6.143045e-08,0.836456,1.340012e-13
2,forward_only,0.460026,1.002998e-03,0.710348,1.557242e-08


In [4]:
baseline_df

,metric,pearson,pearson_p,spearman,spearman_p
0,pagerank,0.459871,0.001007,0.710348,1.557242e-08
1,hub_rank,0.366150,0.010484,0.856155,8.752187e-15
2,authority_rank,0.382106,0.007361,0.845267,4.144001e-14


In [5]:
sensitivity_df.sort_values("spearman", ascending=False).head(10)

,daytime,damping,pearson,pearson_p,spearman,spearman_p
239,0.55,1.00,0.714711,1.158038e-08,0.867323,1.546682e-15
219,0.50,1.00,0.715746,1.078612e-08,0.865528,2.065115e-15
259,0.60,1.00,0.707056,1.940268e-08,0.864742,2.340474e-15
179,0.40,1.00,0.699895,3.098718e-08,0.862609,3.274583e-15
199,0.45,1.00,0.710582,1.532863e-08,0.861487,3.898921e-15
159,0.35,1.00,0.684479,8.117969e-08,0.853573,1.279784e-14
279,0.65,1.00,0.692722,4.886525e-08,0.852507,1.494102e-14
139,0.30,1.00,0.665119,2.513425e-07,0.847512,3.037352e-14
299,0.70,1.00,0.672069,1.691398e-07,0.845099,4.240853e-14
238,0.55,0.95,0.686988,6.968138e-08,0.844706,4.475259e-14


In [6]:
top_scores_df.head(15)

,setting,rank,node,epirank
0,backward_only,1,台北縣板橋市,0.010477
1,backward_only,2,高雄縣鳳山市,0.010332
2,backward_only,3,台北縣中和市,0.008767
3,backward_only,4,高雄市三民區,0.007800
4,backward_only,5,台南市安南區,0.007385
5,backward_only,6,基隆市安樂區,0.006646
6,backward_only,7,花蓮縣吉安鄉,0.006581
7,backward_only,8,台北縣新莊市,0.006299
8,backward_only,9,台北縣土城市,0.006246
9,backward_only,10,台南縣永康市,0.005995


## Generate figures

In [7]:
figures_dir = results_dir / "figures"
figures_dir.mkdir(parents=True, exist_ok=True)
plot.configure_plot_style()

combined_df = pd.concat([
    movement_df.assign(group="EpiRank movement setting"),
    baseline_df.assign(group="Baseline"),
], ignore_index=True)

plot.make_metric_barplot(
    combined_df,
    "spearman",
    "Spearman Correlation Across EpiRank Settings and Baselines",
    figures_dir / "correlation_spearman.png",
)
plot.make_metric_barplot(
    combined_df,
    "pearson",
    "Pearson Correlation Across EpiRank Settings and Baselines",
    figures_dir / "correlation_pearson.png",
)
plot.make_sensitivity_heatmap(
    sensitivity_df,
    "spearman",
    "Sensitivity Heatmap (Spearman)",
    figures_dir / "sensitivity_spearman.png",
)
plot.make_sensitivity_heatmap(
    sensitivity_df,
    "pearson",
    "Sensitivity Heatmap (Pearson)",
    figures_dir / "sensitivity_pearson.png",
)
plot.make_top_scores_plot(
    top_scores_df,
    figures_dir / "top_scores.png",
)

sorted(p.name for p in figures_dir.glob("*.png"))

/Users/gqs/Documents/Net_Dyns/Project/Net_Dyns_Project_EpiRank/scripts/sars_run/plot_sars_results.py:54: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=plot_df, x="metric", y=value_col, palette="deep")
/Users/gqs/Documents/Net_Dyns/Project/Net_Dyns_Project_EpiRank/scripts/sars_run/plot_sars_results.py:54: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(data=plot_df, x="metric", y=value_col, palette="deep")


['correlation_pearson.png',
 'correlation_spearman.png',
 'sensitivity_pearson.png',
 'sensitivity_spearman.png',
 'top_scores.png']